day1-api-extraction/
│
├── extract_customers.ipynb
├── requirements.txt
├── README.md
│
└── output/
    └── customers_clean.csv

# Duka Connect: Customer API Extraction

## Project Overview

Duka Connect requires a clean customer directory from a REST API
so the analytics team can join customer information with sales data.

This notebook will:

1. Retrieve customer records using the requests library.
2. Inspect the nested JSON structure.
3. Extract customer IDs, names, emails, cities, and companies.
4. Build Python lists, tuples, and dictionaries.
5. Flatten nested JSON into a single-level structure.
6. Load the data into a pandas DataFrame.
7. Export a clean CSV file.
8. Handle API failures gracefully.
9. Validate the final output.

**API:** JSONPlaceholder  
**Output:** output/customers_clean.csv

In [1]:
# import libraries
import sys
from pathlib import Path  #	Manages file and folder paths

import pandas as pd
import requests # Communicates with the REST API

In [2]:
# configure the api and output path
API_URL = "https://jsonplaceholder.typicode.com/users"

REQUEST_TIMEOUT = 10

BASE_DIR = Path.cwd()

OUTPUT_PATH = BASE_DIR / "output" / "customers_clean.csv"

print("API URL:", API_URL)
print("Output path:", OUTPUT_PATH)

API URL: https://jsonplaceholder.typicode.com/users
Output path: C:\Users\Solace\Downloads\output\customers_clean.csv


In [3]:
# check current directory
print(Path.cwd())

C:\Users\Solace\Downloads


In [4]:
# API extraction function
class InvalidJSONResponseError(Exception): 
    """Raised when the API response is not valid JSON."""


class InvalidPayloadError(Exception): #to validate that it's a JSON structure
    """Raised when the API returns an unexpected structure."""


def fetch_users(url, timeout=10):
    """Fetch customer records from the API."""

    response = requests.get(url, timeout=timeout)

    # Check the HTTP status before processing the response body.
    response.raise_for_status()

    # Parse the JSON response.
    try:
        payload = response.json()
    except ValueError as exc:
        raise InvalidJSONResponseError(
            "The API returned a response that is not valid JSON."
        ) from exc

    # Validate the response structure.
    if not isinstance(payload, list): #validate the datatype
        raise InvalidPayloadError(
            "Expected a list of customer records."
        )

    if not all(isinstance(user, dict) for user in payload):
        raise InvalidPayloadError(
            "Each customer record must be a dictionary."
        )

    return payload 

In [5]:
# Fetch the customer records

try:
    users = fetch_users(API_URL, timeout=REQUEST_TIMEOUT)

    print(f"Successfully retrieved {len(users)} customer records.")

except requests.exceptions.Timeout:
    print("ERROR: The API request timed out.")

except requests.exceptions.HTTPError as exc:
    status_code = exc.response.status_code if exc.response else "unknown"
    print(f"ERROR: HTTP request failed with status {status_code}.")

except requests.exceptions.RequestException as exc:
    print(f"ERROR: Could not connect to the API: {exc}")

except (InvalidJSONResponseError, InvalidPayloadError) as exc:
    print(f"ERROR: {exc}")

Successfully retrieved 10 customer records.


In [6]:
# structure-inspection function
#validate that you can read each record in the json
def describe_structure(payload):
    """Describe the top-level payload and one record."""

    print("Top-level type:", type(payload).__name__) 

    if not isinstance(payload, list):
        print("Expected a list of records.")
        return

    if not payload:
        print("The API returned an empty list.")
        return

    first_user = payload[0]

    print("Record type:", type(first_user).__name__)

    for key, value in first_user.items():
        print(
            f"Field: {key} | "
            f"Type: {type(value).__name__} | "
            f"Value: {value}"
        )

In [7]:
# API response 
describe_structure(users) 

Top-level type: list
Record type: dict
Field: id | Type: int | Value: 1
Field: name | Type: str | Value: Leanne Graham
Field: username | Type: str | Value: Bret
Field: email | Type: str | Value: Sincere@april.biz
Field: address | Type: dict | Value: {'street': 'Kulas Light', 'suite': 'Apt. 556', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}
Field: phone | Type: str | Value: 1-770-736-8031 x56442
Field: website | Type: str | Value: hildegard.org
Field: company | Type: dict | Value: {'name': 'Romaguera-Crona', 'catchPhrase': 'Multi-layered client-server neural-net', 'bs': 'harness real-time e-markets'}


In [8]:
# Extract the core customer fields

def extract_core_fields(users):
    """Extract the most important customer fields."""

    customers = [] #create an empty list to save data

    for user in users:

        address = user.get("address")
        company = user.get("company")

        # Handle missing or malformed nested objects.
        if not isinstance(address, dict):
            address = {}

        if not isinstance(company, dict):
            company = {}

        customer = {
            "id": user.get("id"),
            "name": user.get("name") or "",
            "email": user.get("email") or "",
            "city": address.get("city") or "",
            "company": company.get("name") or ""
        }

        customers.append(customer)

    return customers

In [9]:
# Execute the function

core_customers = extract_core_fields(users)

print("Number of customers:", len(core_customers))

core_customers[:2]

Number of customers: 10


[{'id': 1,
  'name': 'Leanne Graham',
  'email': 'Sincere@april.biz',
  'city': 'Gwenborough',
  'company': 'Romaguera-Crona'},
 {'id': 2,
  'name': 'Ervin Howell',
  'email': 'Shanna@melissa.tv',
  'city': 'Wisokyburgh',
  'company': 'Deckow-Crist'}]

In [10]:
#  build lists, tuples and dicts

def build_structures(users):
    """Build lists, tuples, and dictionaries from the API records."""

    # List comprehension
    names = [
        user.get("name") or ""
        for user in users
    ]

    # List of tuples
    id_email_pairs = [
        (user.get("id"), user.get("email") or "")
        for user in users
    ]

    # Dictionary comprehension
    id_to_city = {
        user["id"]: (
            user.get("address", {}).get("city") or ""
            if isinstance(user.get("address"), dict)
            else ""
        )
        for user in users
        if user.get("id") is not None
    }

    # Plain for loop
    for name in names:
        print("Customer:", name)

    return names, id_email_pairs, id_to_city

In [11]:
# execute the function

names, id_email_pairs, id_to_city = build_structures(users)

Customer: Leanne Graham
Customer: Ervin Howell
Customer: Clementine Bauch
Customer: Patricia Lebsack
Customer: Chelsey Dietrich
Customer: Mrs. Dennis Schulist
Customer: Kurtis Weissnat
Customer: Nicholas Runolfsdottir V
Customer: Glenna Reichert
Customer: Clementina DuBuque


In [12]:
# get emails
id_email_pairs

[(1, 'Sincere@april.biz'),
 (2, 'Shanna@melissa.tv'),
 (3, 'Nathan@yesenia.net'),
 (4, 'Julianne.OConner@kory.org'),
 (5, 'Lucio_Hettinger@annie.ca'),
 (6, 'Karley_Dach@jasper.info'),
 (7, 'Telly.Hoeger@billy.biz'),
 (8, 'Sherwood@rosamond.me'),
 (9, 'Chaim_McDermott@dana.io'),
 (10, 'Rey.Padberg@karina.biz')]

In [13]:
#  get cities
id_to_city

{1: 'Gwenborough',
 2: 'Wisokyburgh',
 3: 'McKenziehaven',
 4: 'South Elvis',
 5: 'Roscoeview',
 6: 'South Christy',
 7: 'Howemouth',
 8: 'Aliyaview',
 9: 'Bartholomebury',
 10: 'Lebsackbury'}

In [14]:
#  flattening the nested JSON

def flatten_user(user, parent_key="", sep="_"):
    """Flatten nested dictionaries and lists into one dictionary."""

    if not isinstance(user, dict):
        raise TypeError("Each user record must be a dictionary.")

    flattened = {}

    def flatten_value(value, key):

        if isinstance(value, dict):

            if not value:
                flattened[key] = {}

            for child_key, child_value in value.items():

                new_key = (
                    f"{key}{sep}{child_key}"
                    if key
                    else str(child_key)
                )

                flatten_value(child_value, new_key)

        elif isinstance(value, list):

            if not value:
                flattened[key] = []

            for index, item in enumerate(value):

                new_key = (
                    f"{key}{sep}{index}"
                    if key
                    else str(index)
                )

                flatten_value(item, new_key)

        else:
            flattened[key] = value

    for key, value in user.items():

        new_key = (
            f"{parent_key}{sep}{key}"
            if parent_key
            else str(key)
        )

        flatten_value(value, new_key)

    return flattened

In [15]:
#  flatten customer data/records

flat_users = [
    flatten_user(user)
    for user in users
]

print("Original record:")
print(users[0])

print("\nFlattened record:")
print(flat_users[0])

Original record:
{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'address': {'street': 'Kulas Light', 'suite': 'Apt. 556', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}, 'phone': '1-770-736-8031 x56442', 'website': 'hildegard.org', 'company': {'name': 'Romaguera-Crona', 'catchPhrase': 'Multi-layered client-server neural-net', 'bs': 'harness real-time e-markets'}}

Flattened record:
{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'address_street': 'Kulas Light', 'address_suite': 'Apt. 556', 'address_city': 'Gwenborough', 'address_zipcode': '92998-3874', 'address_geo_lat': '-37.3159', 'address_geo_lng': '81.1496', 'phone': '1-770-736-8031 x56442', 'website': 'hildegard.org', 'company_name': 'Romaguera-Crona', 'company_catchPhrase': 'Multi-layered client-server neural-net', 'company_bs': 'harness real-time e-markets'}


In [16]:
#  create a pandas df

def to_dataframe(flat_users):
    """Convert flattened customer records into a clean DataFrame."""

    df = pd.DataFrame(flat_users)

    # Rename the customer identifier.
    df = df.rename(columns={"id": "customer_id"})

    # Convert geographic coordinates to numeric values.
    coordinate_columns = [
        "address_geo_lat",
        "address_geo_lng"
    ]

    for column in coordinate_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(
                df[column],
                errors="coerce"
            )

    # Put the main customer fields first.
    preferred_columns = [
        "customer_id",
        "name",
        "email",
        "address_city",
        "company_name",
        "address_geo_lat",
        "address_geo_lng"
    ]

    ordered_columns = [
        column
        for column in preferred_columns
        if column in df.columns
    ]

    remaining_columns = [
        column
        for column in df.columns
        if column not in ordered_columns
    ]

    df = df[ordered_columns + remaining_columns]

    print("DataFrame shape:", df.shape)

    print("\nData types:")
    print(df.dtypes)

    return df

In [17]:
#  inspect the data
df = to_dataframe(flat_users)

DataFrame shape: (10, 15)

Data types:
customer_id              int64
name                       str
email                      str
address_city               str
company_name               str
address_geo_lat        float64
address_geo_lng        float64
username                   str
address_street             str
address_suite              str
address_zipcode            str
phone                      str
website                    str
company_catchPhrase        str
company_bs                 str
dtype: object


In [18]:
#  view the data
df.head()

,customer_id,name,email,address_city,company_name,address_geo_lat,address_geo_lng,username,address_street,address_suite,address_zipcode,phone,website,company_catchPhrase,company_bs
0,1,Leanne Graham,Sincere@april.biz,Gwenborough,Romaguera-Crona,-37.3159,81.1496,Bret,Kulas Light,Apt. 556,92998-3874,1-770-736-8031 x56442,hildegard.org,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Shanna@melissa.tv,Wisokyburgh,Deckow-Crist,-43.9509,-34.4618,Antonette,Victor Plains,Suite 879,90566-7771,010-692-6593 x09125,anastasia.net,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Nathan@yesenia.net,McKenziehaven,Romaguera-Jacobson,-68.6102,-47.0653,Samantha,Douglas Extension,Suite 847,59590-4157,1-463-123-4447,ramiro.info,Face to face bifurcated interface,e-enable strategic applications
3,4,Patricia Lebsack,Julianne.OConner@kory.org,South Elvis,Robel-Corkery,29.4572,-164.2990,Karianne,Hoeger Mall,Apt. 692,53919-4257,493-170-9623 x156,kale.biz,Multi-tiered zero tolerance productivity,transition cutting-edge web services
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca,Roscoeview,Keebler LLC,-31.8129,62.5342,Kamren,Skiles Walks,Suite 351,33263,(254)954-1289,demarco.info,User-centric fault-tolerant solution,revolutionize end-to-end systems


In [20]:
#  export the csv
output_path = "customers_clean.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("CSV exported successfully:", output_path)
print("Rows exported:", len(df))

CSV exported successfully: customers_clean.csv
Rows exported: 10


In [22]:
# confirm if csv was exported
df_check = pd.read_csv(output_path)

print(df_check.head(2))
print("Rows in exported CSV:", len(df_check))

   customer_id              name                      email   address_city  \
0            1     Leanne Graham          Sincere@april.biz    Gwenborough   
1            2      Ervin Howell          Shanna@melissa.tv    Wisokyburgh   
2            3  Clementine Bauch         Nathan@yesenia.net  McKenziehaven   
3            4  Patricia Lebsack  Julianne.OConner@kory.org    South Elvis   
4            5  Chelsey Dietrich   Lucio_Hettinger@annie.ca     Roscoeview   

         company_name  address_geo_lat  address_geo_lng   username  \
0     Romaguera-Crona         -37.3159          81.1496       Bret   
1        Deckow-Crist         -43.9509         -34.4618  Antonette   
2  Romaguera-Jacobson         -68.6102         -47.0653   Samantha   
3       Robel-Corkery          29.4572        -164.2990   Karianne   
4         Keebler LLC         -31.8129          62.5342     Kamren   

      address_street address_suite address_zipcode                  phone  \
0        Kulas Light      Apt. 55